In [1]:
!pip install -q optuna xgboost lightgbm catboost

In [2]:
# Imports
import os, sys, time, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import TimeSeriesSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
PRICE_FILE = DRIVE_ROOT / 'Local_Price.csv'
WEEKLY_DIR = DRIVE_ROOT / 'Weekly_Data'
OUT_DIR = DRIVE_ROOT / 'model_outputs'
OUT_DIR.mkdir(exist_ok=True, parents=True)

In [5]:
# Basic utils
def safe_parse_date(s, dayfirst=False):
    return pd.to_datetime(s, dayfirst=dayfirst, errors='coerce')

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    denom = np.where(np.abs(y_true) < 1e-9, 1e-9, y_true)
    mape = np.mean(np.abs((y_true - y_pred) / denom)) * 100
    accuracy = 100.0 - mape
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape, 'Accuracy%': accuracy}

def savefig(fig, fname):
    path = OUT_DIR / fname
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)

In [6]:
# Load datasets
print("Loading price data...")
price_df = pd.read_csv(PRICE_FILE)
if 'Date' not in price_df.columns:
    raise ValueError("Local_Price.csv must contain a 'Date' column.")
# parse Date: user said mm/dd/yyyy
price_df['Date'] = safe_parse_date(price_df['Date'], dayfirst=False)
price_df['District'] = price_df['District'].astype(str).str.strip()

# Load all weekly files into dict
weather_by_district = {}
for f in sorted(WEEKLY_DIR.glob('weekly_*.csv')):
    name = f.stem.replace('weekly_', '')
    try:
        w = pd.read_csv(f)
        # normalize column names
        w.columns = [c.strip() for c in w.columns]
        # parse Week as mm/dd/yyyy (month first)
        if 'Week' in w.columns:
            w['Week'] = safe_parse_date(w['Week'], dayfirst=False)
        else:
            # fallback: parse first column
            w.iloc[:,0] = safe_parse_date(w.iloc[:,0], dayfirst=False)
            w.rename(columns={w.columns[0]: 'Week'}, inplace=True)
        # lowercase column names for weather features
        w = w.rename(columns={c: c.strip().lower().replace(' ', '_') for c in w.columns})
        weather_by_district[name] = w
    except Exception as e:
        print(f"Warning: Could not load or process {f.name} due to error: {e}")

print("Loaded weather files:", list(weather_by_district.keys()))
print("Price rows:", len(price_df))

Loading price data...
Loaded weather files: ['Badulla', 'Colombo', 'Galle', 'Gampaha', 'Hambantota', 'Kalutara', 'Kandy', 'Kegalle', 'Kurunegala', 'Matale', 'Matara', 'Monaragala', 'NuwaraEliya', 'Ratnapura']
Price rows: 6821


In [7]:
# Configurations
TARGET_COLUMNS = [
    "Black GR-1 (Average Price)",
    "Black GR-2 (Average Price)",
    "WHITE (Average Price)"
]
# Which target do you want to run the whole pipeline for now?
# (We will train/aggregate across districts for the chosen target,
#  you can loop over TARGET_COLUMNS later if you want all three.)
TARGET_COL = "Black GR-1 (Average Price)"   # <- change to other allowed values as needed

# Tuning / runtime controls (increase for better performance)
TEST_WEEKS = 52                 # per-district holdout weeks
N_TRIALS_PER_DISTRICT = 20      # Optuna trials per model per district
TS_SPLITS = 3                   # TimeSeriesSplit folds inside Optuna objective
GLOBAL_OPTUNA_TRIALS = 40       # Optuna trials for final global model

# sanity checks
if TARGET_COL not in TARGET_COLUMNS:
    raise ValueError(f"TARGET_COL must be one of {TARGET_COLUMNS}")

In [8]:
# Preprocess price_df and weather_by_district (missing handling)
# Clean district strings
price_df['District'] = price_df['District'].astype(str).str.strip()

# Ensure target exists (case sensitive by name)
if TARGET_COL not in price_df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not present in Local_Price.csv. Available columns: {list(price_df.columns)}")

# Drop rows missing key identifiers
price_df = price_df.dropna(subset=['Date', 'District']).reset_index(drop=True)

# Convert known price columns to numeric (coerce)
for col in TARGET_COLUMNS:
    if col in price_df.columns:
        # Remove commas before converting to numeric
        price_df[col] = price_df[col].astype(str).str.replace(',', '', regex=False)
        price_df[col] = pd.to_numeric(price_df[col], errors='coerce')

# Fill missing price values per-district using median (robust)
for col in TARGET_COLUMNS:
    if col in price_df.columns:
        price_df[col] = price_df.groupby('District')[col].transform(lambda s: s.fillna(s.median()))

# Normalize weather_by_district: ensure expected weather columns exist and apply ffill/bfill per-district
WEATHER_FEATURES = ['temperature', 'relative_humidity', 'rainfall', 'wind_speed', 'soil_temperature', 'soil_moisture']

for d, w in list(weather_by_district.items()):
    # ensure 'week' column present (we already parsed earlier)
    if 'week' not in w.columns:
        # try to rename first column if not present
        w.rename(columns={w.columns[0]: 'week'}, inplace=True)
    # convert weather columns to numeric and ffill/bfill
    for col in WEATHER_FEATURES:
        if col in w.columns:
            w[col] = pd.to_numeric(w[col], errors='coerce')
            w[col] = w.groupby('week')[col].transform(lambda s: s)  # noop but keeps shape
            # For each district DataFrame we forward/backfill
            w[col] = w[col].fillna(method='ffill').fillna(method='bfill')
    # save cleaned df back
    weather_by_district[d] = w

/tmp/ipython-input-2044767209.py:38: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  w[col] = w[col].fillna(method='ffill').fillna(method='bfill')


In [9]:
# Helper: build merged dataset for one district
def build_district_df(district, target_col=TARGET_COL, min_rows=TEST_WEEKS+20):
    """
    Returns merged DataFrame for district with engineered features and lag features.
    Drops initial rows without lags.
    """
    print(f"  [DEBUG] Processing district: {district}")

    # filter price rows for this district
    p = price_df[price_df['District'].str.lower() == district.lower()].copy()
    if p.empty:
        raise ValueError(f"No price rows for district '{district}'")
    print(f"  [DEBUG] Length of price_df for {district} before target selection: {len(p)}")

    # ensure target exists in p (case-insensitive search)
    target_match = None
    for c in p.columns:
        if c.lower() == target_col.lower():
            target_match = c
            break
    if target_match is None:
        # try substring match
        for c in p.columns:
            if target_col.lower() in c.lower():
                target_match = c
                break
    if target_match is None:
        raise ValueError(f"Target '{target_col}' not found in price columns for district {district}")
    # keep Date and price
    p = p[['Date', target_match]].rename(columns={target_match: 'price'})
    print(f"  [DEBUG] Length of price_df for {district} after target selection: {len(p)}")

    # Add a 'week_start' column to price_df for merging with weekly weather data
    p['week_start'] = p['Date'].dt.to_period('W').dt.start_time

    # load weather df for district
    # First, try direct match
    w = weather_by_district.get(district)
    if w is None:
        # If no direct match, normalize district name for lookup in weather_by_district
        normalized_district_price_df = district.replace(' ', '').replace('_', '')
        w = weather_by_district.get(normalized_district_price_df)
        if w is None:
            # As a fallback, try case-insensitive and space/underscore-insensitive match across all keys
            for k in weather_by_district:
                if k.lower().replace(' ', '').replace('_', '') == district.lower().replace(' ', '').replace('_', ''):
                    w = weather_by_district[k]
                    break
    if w is None:
        raise ValueError(f"No weather file for district '{district}'")

    # copy and ensure week col name 'week'
    w = w.copy()
    if 'week' not in w.columns:
        w.rename(columns={w.columns[0]: 'week'}, inplace=True)

    # merge on Date==week_start (from price_df) and week (from weather_df)
    merged = pd.merge(p, w, left_on='week_start', right_on='week', how='left', suffixes=('','_w'))
    print(f"  [DEBUG] Length of merged_df after initial merge: {len(merged)}")

    # fill weather gaps using neighborhood (ffill/bfill)
    valid_weather_cols = []
    for col in WEATHER_FEATURES:
        if col in merged.columns:
            merged[col] = pd.to_numeric(merged[col], errors='coerce')
            merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
            if not merged[col].isnull().all(): # Only keep if not entirely NaN after imputation
                valid_weather_cols.append(col)
            else:
                print(f"  [DEBUG] Dropping completely NaN weather column: {col} for {district}")
                merged = merged.drop(columns=[col])

    # drop rows with missing price
    merged = merged.dropna(subset=['price']).sort_values('Date').reset_index(drop=True)
    print(f"  [DEBUG] Length of merged_df after dropping rows with missing price: {len(merged)}")

    # if dataset too short, return None
    if len(merged) < min_rows:
        print(f"  [DEBUG] Merged_df too short ({len(merged)} rows) for {district} before feature engineering.")
        return merged # will be skipped by caller

    # Feature engineering: lags (1..12), rolling means, time features
    for lag in range(1, 13):
        merged[f'price_lag_{lag}'] = merged['price'].shift(lag)
    merged['rolling_3']  = merged['price'].rolling(window=3, min_periods=1).mean()
    merged['rolling_6']  = merged['price'].rolling(window=6, min_periods=1).mean()
    merged['rolling_12'] = merged['price'].rolling(window=12, min_periods=1).mean()

    merged['year'] = merged['Date'].dt.year
    merged['month'] = merged['Date'].dt.month
    merged['weekofyear'] = merged['Date'].dt.isocalendar().week.astype(int)
    merged['sin_month'] = np.sin(2*np.pi*merged['month']/12)
    merged['cos_month'] = np.cos(2*np.pi*merged['month']/12)
    merged['time_index'] = np.arange(len(merged))
    print(f"  [DEBUG] Length of merged_df after feature engineering (before final dropna): {len(merged)}")

    # Drop initial rows with NaN in lag features (first 12 rows) and then any other NaNs in feature columns
    # More robust than a blanket dropna() after lags are introduced
    merged = merged.iloc[12:]
    print(f"  [DEBUG] Length of merged_df after dropping initial 12 rows (for lags): {len(merged)}")

    # Define all relevant features for final NaN check
    engineered_features = ['time_index','year','month','weekofyear','sin_month','cos_month','rolling_3','rolling_6','rolling_12'] + [f'price_lag_{i}' for i in range(1,13)]
    all_numeric_features = [c for c in (valid_weather_cols + engineered_features) if c in merged.columns]

    # Drop NaNs only from the relevant feature and target columns
    merged = merged.dropna(subset=['price'] + all_numeric_features).reset_index(drop=True)
    print(f"  [DEBUG] Length of merged_df after final dropna on subset: {len(merged)}")

    return merged

In [10]:
# Optuna objective factory per model family
def get_optuna_objective(model_key, X, y, tscv):
    def objective(trial):
        if model_key == 'rf':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
                'max_depth': trial.suggest_int('max_depth', 6, 40),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
                'bootstrap': trial.suggest_categorical('bootstrap', [True, False])
            }
            model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)

        elif model_key == 'xgb':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 400, 2000),
                'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
                'max_depth': trial.suggest_int('max_depth', 4, 12),
                'min_child_weight': trial.suggest_float('min_child_weight', 1, 10),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'gamma': trial.suggest_float('gamma', 0, 5),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True)
            }
            model = XGBRegressor(objective='reg:squarederror', **params, random_state=42, n_jobs=-1)

        elif model_key == 'lgbm':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 300, 2000),
                'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 31, 500),
                'max_depth': trial.suggest_int('max_depth', -1, 50),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
                'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
                'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 2.0),
                'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 2.0)
            }
            model = LGBMRegressor(**params, random_state=42, n_jobs=-1)

        elif model_key == 'cat':
            params = {
                'iterations': trial.suggest_int('iterations', 500, 3000),
                'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
                'depth': trial.suggest_int('depth', 4, 12),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 10, log=True),
                'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 10.0),
                'random_strength': trial.suggest_float('random_strength', 1.0, 50.0)

            }
            model = CatBoostRegressor(**params, verbose=0, random_seed=42)

        else:
            raise ValueError("Unknown model key for Optuna objective")

        # CV: use median imputer internally
        imputer = SimpleImputer(strategy='median')
        maes = []
        Xc = X.copy()
        for tr_idx, val_idx in tscv.split(Xc):
            Xtr, Xv = Xc.iloc[tr_idx], Xc.iloc[val_idx]
            ytr, yv = y.iloc[tr_idx], y.iloc[val_idx]
            Xtr_imp = pd.DataFrame(imputer.fit_transform(Xtr), columns=Xtr.columns, index=Xtr.index)
            Xv_imp  = pd.DataFrame(imputer.transform(Xv), columns=Xv.columns, index=Xv.index)
            model.fit(Xtr_imp, ytr)
            preds = model.predict(Xv_imp)
            maes.append(mean_absolute_error(yv, preds))
        return float(np.mean(maes))
    return objective

In [11]:
import os
from pathlib import Path

WEEKLY_DIR = Path('/content/drive/MyDrive/Weekly_Data')

if WEEKLY_DIR.exists():
    print(f"Files in {WEEKLY_DIR}:")
    for f in sorted(WEEKLY_DIR.glob('weekly_*.csv')):
        print(f.name)
else:
    print(f"Directory not found: {WEEKLY_DIR}")

Files in /content/drive/MyDrive/Weekly_Data:
weekly_Badulla.csv
weekly_Colombo.csv
weekly_Galle.csv
weekly_Gampaha.csv
weekly_Hambantota.csv
weekly_Kalutara.csv
weekly_Kandy.csv
weekly_Kegalle.csv
weekly_Kurunegala.csv
weekly_Matale.csv
weekly_Matara.csv
weekly_Monaragala.csv
weekly_NuwaraEliya.csv
weekly_Ratnapura.csv


In [12]:
# District-wise loop: tune + train + predict on holdout (collect for aggregation)
DISTRICTS = sorted([d for d in price_df['District'].unique() if isinstance(d, str)])
print("Districts:", DISTRICTS)

# prepared containers for aggregated predictions per family
agg = {k: {'y_true': [], 'y_pred': []} for k in ['rf','xgb','lgbm','cat']}
per_district_records = []

start_time = time.time()
for district in DISTRICTS:
    try:
        print("\n----- District:", district, "-----")
        merged = build_district_df(district, TARGET_COL)
        if len(merged) < (TEST_WEEKS + 20):
            print(f"  skipping {district} due to insufficient rows: {len(merged)}")
            continue

        # Identify features: weather + engineered + lags
        weather_cols_existing = [c for c in WEATHER_FEATURES if c in merged.columns]
        engineered = ['time_index','year','month','weekofyear','sin_month','cos_month','rolling_3','rolling_6','rolling_12'] + [f'price_lag_{i}' for i in range(1,13)]
        feature_cols = [c for c in weather_cols_existing + engineered if c in merged.columns]

        # Train/test split by time
        train_df = merged.iloc[:-TEST_WEEKS].reset_index(drop=True)
        test_df  = merged.iloc[-TEST_WEEKS:].reset_index(drop=True)

        X_train = train_df[feature_cols].copy()
        y_train = train_df['price'].copy()
        X_test  = test_df[feature_cols].copy()
        y_test  = test_df['price'].copy()

        tscv = TimeSeriesSplit(n_splits=TS_SPLITS)

        # For each model family tune & fit
        for model_key in ['rf','xgb','lgbm','cat']:
            print(f"  tuning {model_key} ...", end=' ')
            obj = get_optuna_objective(model_key, X_train, y_train, tscv)
            study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
            study.optimize(obj, n_trials=N_TRIALS_PER_DISTRICT, show_progress_bar=False)

            print(f"done (best_cv_mae={study.best_value:.3f})")
            best_params = study.best_params

            # build final model
            if model_key == 'rf':
                model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
            elif model_key == 'xgb':
                model = XGBRegressor(objective='reg:squarederror', **best_params, random_state=42, n_jobs=-1)
            elif model_key == 'lgbm':
                model = LGBMRegressor(**best_params, random_state=42, n_jobs=-1)
            elif model_key == 'cat':
                model = CatBoostRegressor(**best_params, verbose=0, random_seed=42)

            # impute and fit
            imputer = SimpleImputer(strategy='median')
            Xtr_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
            Xte_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

            model.fit(Xtr_imp, y_train)
            preds = model.predict(Xte_imp)

            # aggregate predictions for global evaluation
            agg[model_key]['y_true'].extend(list(y_test.values))
            agg[model_key]['y_pred'].extend(list(preds))

            # save per-district model bundle
            joblib.dump({'model': model, 'imputer': imputer, 'feature_cols': feature_cols},
                        OUT_DIR / f"{district}_{model_key}.joblib")

            # record per-district metrics
            record = evaluate(y_test.values, preds)
            record.update({'district': district, 'model': model_key, 'best_cv_mae': study.best_value})
            per_district_records.append(record)

            # save diagnostic plot (pred vs actual)
            fig, ax = plt.subplots(figsize=(8,3))
            ax.plot(test_df['Date'], y_test.values, label='Actual', marker='o')
            ax.plot(test_df['Date'], preds, label='Predicted', marker='x')
            ax.set_title(f"{district} - {model_key.upper()} (test)")
            ax.legend()
            plt.xticks(rotation=30)
            savefig(fig, f"pred_vs_actual_{district}_{model_key}.png")

    except Exception as e:
        print("Error on district", district, ":", e)
        traceback.print_exc()
        continue

elapsed = time.time() - start_time
print(f"\nDistrict-wise tuning done in {elapsed/60:.1f} minutes (quick-run)")

Districts: ['Badulla', 'Colombo', 'Galle', 'Gampaha', 'Hambantota', 'Kalutara', 'Kandy', 'Kegalle', 'Kurunegala', 'Matale', 'Matara', 'Monaragala', 'Nuwara Eliya', 'Ratnapura']

----- District: Badulla -----
  [DEBUG] Processing district: Badulla
  [DEBUG] Length of price_df for Badulla before target selection: 189
  [DEBUG] Length of price_df for Badulla after target selection: 189
  [DEBUG] Length of merged_df after initial merge: 189
  [DEBUG] Dropping completely NaN weather column: temperature for Badulla
  [DEBUG] Dropping completely NaN weather column: relative_humidity for Badulla
  [DEBUG] Dropping completely NaN weather column: rainfall for Badulla
  [DEBUG] Dropping completely NaN weather column: wind_speed for Badulla
  [DEBUG] Dropping completely NaN weather column: soil_temperature for Badulla
  [DEBUG] Dropping completely NaN weather column: soil_moisture for Badulla
  [DEBUG] Length of merged_df after dropping rows with missing price: 189
  [DEBUG] Length of merged_df af

/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-30533

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [W

/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-30533

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [W

...

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [W

/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-30533

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [W

/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-30533

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [W

In [13]:
# Compute global aggregated metrics per model family
global_metrics = {}
for key in ['rf','xgb','lgbm','cat']:
    y_true = np.array(agg[key]['y_true'])
    y_pred = np.array(agg[key]['y_pred'])
    if len(y_true) == 0:
        continue
    global_metrics[key] = evaluate(y_true, y_pred)

print("\nGlobal aggregated metrics (across all district tests):")
gm_df = pd.DataFrame(global_metrics).T
display(gm_df)

# Save global metrics and per-district metrics
gm_df.to_csv(OUT_DIR / "global_metrics_by_model_family.csv")
pd.DataFrame(per_district_records).to_csv(OUT_DIR / "per_district_model_metrics.csv", index=False)


Global aggregated metrics (across all district tests):


,MAE,RMSE,R2,MAPE,Accuracy%
rf,234.018608,290.820391,-0.903985,12.649555,87.350445
xgb,226.169739,270.179222,-0.643303,12.124924,87.875076
lgbm,224.021499,267.576125,-0.611790,12.027519,87.972481
cat,259.560063,308.886241,-1.147885,13.856036,86.143964


In [14]:
# Choose single best model family (lowest global MAE) and train final global model
best_family = min(global_metrics.items(), key=lambda kv: kv[1]['MAE'])[0]
family_name_map = {'rf':'RandomForest','xgb':'XGBoost','lgbm':'LightGBM','cat':'CatBoost'}
print(f"\nBest model family overall (by lowest MAE): {family_name_map[best_family]} ({best_family})")

# Build combined dataset across districts and keep district dummies
combined_list = []
for district in DISTRICTS:
    try:
        df = build_district_df(district, TARGET_COL, min_rows=TEST_WEEKS+1)
        if df is None or len(df) < TEST_WEEKS+10:
            continue
        df['district'] = district
        combined_list.append(df)
    except Exception:
        continue

if len(combined_list) == 0:
    raise RuntimeError("No combined data available. Check your datasets or reduce TEST_WEEKS.")

combined_df = pd.concat(combined_list, ignore_index=True)
combined_df = combined_df.sort_values('Date').reset_index(drop=True)

# define global feature columns
weather_cols_present = [c for c in WEATHER_FEATURES if c in combined_df.columns]
engineered = ['time_index','year','month','weekofyear','sin_month','cos_month','rolling_3','rolling_6','rolling_12'] + [f'price_lag_{i}' for i in range(1,13)]
core_features = [c for c in weather_cols_present + engineered if c in combined_df.columns]

# one-hot districts
combined_df = pd.get_dummies(combined_df, columns=['district'], prefix='district')
district_dummies = [c for c in combined_df.columns if c.startswith('district_')]
feature_cols_global = core_features + district_dummies

# drop rows with NaNs in features or price
combined_df = combined_df.dropna(subset=feature_cols_global + ['price']).reset_index(drop=True)


Best model family overall (by lowest MAE): LightGBM (lgbm)
  [DEBUG] Processing district: Badulla
  [DEBUG] Length of price_df for Badulla before target selection: 189
  [DEBUG] Length of price_df for Badulla after target selection: 189
  [DEBUG] Length of merged_df after initial merge: 189
  [DEBUG] Dropping completely NaN weather column: temperature for Badulla
  [DEBUG] Dropping completely NaN weather column: relative_humidity for Badulla
  [DEBUG] Dropping completely NaN weather column: rainfall for Badulla
  [DEBUG] Dropping completely NaN weather column: wind_speed for Badulla
  [DEBUG] Dropping completely NaN weather column: soil_temperature for Badulla
  [DEBUG] Dropping completely NaN weather column: soil_moisture for Badulla
  [DEBUG] Length of merged_df after dropping rows with missing price: 189
  [DEBUG] Length of merged_df after feature engineering (before final dropna): 189
  [DEBUG] Length of merged_df after dropping initial 12 rows (for lags): 177
  [DEBUG] Length of 

/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-30533

  [DEBUG] Length of merged_df after dropping rows with missing price: 189
  [DEBUG] Length of merged_df after feature engineering (before final dropna): 189
  [DEBUG] Length of merged_df after dropping initial 12 rows (for lags): 177
  [DEBUG] Length of merged_df after final dropna on subset: 177
  [DEBUG] Processing district: Matara
  [DEBUG] Length of price_df for Matara before target selection: 189
  [DEBUG] Length of price_df for Matara after target selection: 189
  [DEBUG] Length of merged_df after initial merge: 189
  [DEBUG] Dropping completely NaN weather column: temperature for Matara
  [DEBUG] Dropping completely NaN weather column: relative_humidity for Matara
  [DEBUG] Dropping completely NaN weather column: rainfall for Matara
  [DEBUG] Dropping completely NaN weather column: wind_speed for Matara
  [DEBUG] Dropping completely NaN weather column: soil_temperature for Matara
  [DEBUG] Dropping completely NaN weather column: soil_moisture for Matara
  [DEBUG] Length of merge

/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-30533

In [15]:
# global train/test split (hold last GLOBAL_TEST_WEEKS)
GLOBAL_TEST_WEEKS = 52
if len(combined_df) <= GLOBAL_TEST_WEEKS + 10:
    raise RuntimeError("Not enough combined rows for global test; reduce GLOBAL_TEST_WEEKS or check data.")
train_global = combined_df.iloc[:-GLOBAL_TEST_WEEKS].reset_index(drop=True)
test_global  = combined_df.iloc[-GLOBAL_TEST_WEEKS:].reset_index(drop=True)

Xg_train = train_global[feature_cols_global].copy()
yg_train = train_global['price'].copy()
Xg_test  = test_global[feature_cols_global].copy()
yg_test  = test_global['price'].copy()

# Optuna tune the chosen family on combined dataset (TimeSeries CV)
print(f"\nTuning final global {best_family} model with Optuna (trials={GLOBAL_OPTUNA_TRIALS}) ...")
tscv_global = TimeSeriesSplit(n_splits=3)
obj_global = get_optuna_objective(best_family, Xg_train, yg_train, tscv_global)
study_global = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_global.optimize(obj_global, n_trials=GLOBAL_OPTUNA_TRIALS, show_progress_bar=True)
print("Best global params:", study_global.best_params)


Tuning final global lgbm model with Optuna (trials=40) ...


  0%|          | 0/40 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

In [16]:
# build final global model
global_best_params = study_global.best_params
if best_family == 'rf':
    final_global_model = RandomForestRegressor(**global_best_params, random_state=42, n_jobs=-1)
elif best_family == 'xgb':
    final_global_model = XGBRegressor(objective='reg:squarederror', **global_best_params, random_state=42, n_jobs=-1)
elif best_family == 'lgbm':
    final_global_model = LGBMRegressor(**global_best_params, random_state=42, n_jobs=-1)
elif best_family == 'cat':
    final_global_model = CatBoostRegressor(**global_best_params, verbose=0, random_seed=42)

# impute global features
global_imputer = SimpleImputer(strategy='median')
Xg_train_imp = pd.DataFrame(global_imputer.fit_transform(Xg_train), columns=Xg_train.columns)
Xg_test_imp  = pd.DataFrame(global_imputer.transform(Xg_test), columns=Xg_test.columns)

final_global_model.fit(Xg_train_imp, yg_train)
yg_pred = final_global_model.predict(Xg_test_imp)
final_global_metrics = evaluate(yg_test.values, yg_pred)
print("\nFinal global model test metrics:")
display(pd.DataFrame([final_global_metrics], index=[family_name_map[best_family]]).T)

# Save final model bundle
joblib.dump({
    'model_family': best_family,
    'model': final_global_model,
    'imputer': global_imputer,
    'feature_cols': feature_cols_global
}, OUT_DIR / f"final_global_{family_name_map[best_family]}.joblib")
print("Saved final global model:", OUT_DIR / f"final_global_{family_name_map[best_family]}.joblib")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000785 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4110
[LightGBM] [Info] Number of data points in the train set: 2426, number of used features: 35
[LightGBM] [Info] Start training from score 1125.974237
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positiv

,LightGBM
MAE,32.038373
RMSE,41.690911
R2,0.427450
MAPE,1.598814
Accuracy%,98.401186


Saved final global model: /content/drive/MyDrive/model_outputs/final_global_LightGBM.joblib


In [17]:
# Save CSVs and a simple comparison plot
pd.DataFrame(global_metrics).T.to_csv(OUT_DIR / "global_metrics_by_model_family.csv")
pd.DataFrame(per_district_records).to_csv(OUT_DIR / "per_district_model_metrics.csv", index=False)
pd.DataFrame([final_global_metrics]).T.to_csv(OUT_DIR / "final_global_model_metrics.csv")

# Model comparison bar (MAE & RMSE)
cmp_df = pd.DataFrame(global_metrics).T.reset_index().rename(columns={'index':'model'})
if not cmp_df.empty:
    fig, ax = plt.subplots(figsize=(8,4))
    sns.barplot(data=cmp_df.melt(id_vars='model', value_vars=['MAE','RMSE']), x='model', y='value', hue='variable', ax=ax)
    plt.title("Global model family comparison")
    savefig(fig, "global_model_family_comparison.png")

# Plot predicted vs actual for final global model
fig2, ax2 = plt.subplots(figsize=(10,4))
ax2.plot(test_global['Date'], yg_test.values, label='Actual')
ax2.plot(test_global['Date'], yg_pred, label='Predicted')
ax2.set_title(f"Final global {family_name_map[best_family]} - Actual vs Predicted")
ax2.legend()
plt.xticks(rotation=30)
savefig(fig2, "final_global_pred_vs_actual.png")

print("\nPipeline finished. Outputs saved in:", OUT_DIR)


Pipeline finished. Outputs saved in: /content/drive/MyDrive/model_outputs


In [18]:
# Prediction helper (uses the saved final global model)
def predict_price_global(district, pepper_type, grade, year, month, week_iso=None, weather_values=None):
    """
    district: exact district name string matching your weekly files (e.g. 'Colombo')
    pepper_type, grade: informational only (we used TARGET_COL); keep consistent with training
    year, month: ints
    week_iso: optional ISO week number (1-53); if None we map to first day of month
    weather_values: optional dict to override weather features, e.g. {'temperature': 27.0, 'rainfall': 12.3}
    """
    # load final model bundle
    bundle_path = OUT_DIR / f"final_global_{family_name_map[best_family]}.joblib"
    bundle = joblib.load(bundle_path)
    model = bundle['model']
    imputer = bundle['imputer']
    feature_cols = bundle['feature_cols']

    # Build a single-row input using last known lags from district
    merged = build_district_df(district, TARGET_COL, min_rows=10)
    if merged is None or merged.empty:
        raise ValueError("No district data available to compute lag features for prediction.")

    # map year+week_iso to a date (Monday)
    if week_iso is not None:
        try:
            date = pd.to_datetime(f'{year}-W{int(week_iso)}-1', format='%G-W%V-%u')
        except Exception:
            date = pd.Timestamp(year=year, month=month, day=1)
    else:
        date = pd.Timestamp(year=year, month=month, day=1)

    # attempt to fetch weather row for the date
    wdf = weather_by_district.get(district)
    if wdf is None:
        # try case-insensitive
        for k in weather_by_district:
            if k.lower() == district.lower():
                wdf = weather_by_district[k]; break
    # find matching week row
    weather_row = None
    if wdf is not None:
        wdf_tmp = wdf.copy()
        wdf_tmp['year'] = wdf_tmp['week'].dt.year
        wdf_tmp['weekofyear'] = wdf_tmp['week'].dt.isocalendar().week.astype(int)
        yw = date.isocalendar()[1]
        candidate = wdf_tmp[(wdf_tmp['year']==date.year) & (wdf_tmp['weekofyear']==yw)]
        if len(candidate) > 0:
            weather_row = candidate.iloc[0]
        else:
            weather_row = wdf_tmp.sort_values('week').iloc[-1]

    # prepare base row using last known lags
    last = merged.sort_values('Date').iloc[-1]
    base = {}
    # weather features
    for col in weather_cols_present:
        if weather_values and col in weather_values:
            base[col] = weather_values[col]
        elif weather_row is not None and col in weather_row.index:
            base[col] = float(weather_row[col])
        else:
            base[col] = float(last.get(col, 0.0))
    # time features
    base['year'] = date.year
    base['month'] = date.month
    base['weekofyear'] = date.isocalendar()[1]
    base['sin_month'] = np.sin(2*np.pi*base['month']/12)
    base['cos_month'] = np.cos(2*np.pi*base['month']/12)
    base['time_index'] = len(merged) + 1
    # lags and rolling features from last row
    for lag in range(1,13):
        col = f'price_lag_{lag}'
        base[col] = last.get(col, last['price'])
    base['rolling_3']  = last.get('rolling_3', last['price'])
    base['rolling_6']  = last.get('rolling_6', last['price'])
    base['rolling_12'] = last.get('rolling_12', last['price'])

    # district dummies - set 1 for requested district, 0 otherwise
    for dcol in district_dummies:
        base[dcol] = 1.0 if dcol == f"district_{district}" else 0.0

    # build DataFrame and ensure feature order
    X_single = pd.DataFrame([ {c: base.get(c, 0.0) for c in feature_cols} ])
    X_single_imp = pd.DataFrame(imputer.transform(X_single), columns=X_single.columns)

    pred = model.predict(X_single_imp)
    return float(pred[0])

In [21]:
# Example usage:
pred = predict_price_global('Badulla', 'Black', 'GR-1 (Average Price)', 2025, 11, week_iso=45)
print("Predicted price:", pred)

  [DEBUG] Processing district: Badulla
  [DEBUG] Length of price_df for Badulla before target selection: 189
  [DEBUG] Length of price_df for Badulla after target selection: 189
  [DEBUG] Length of merged_df after initial merge: 189
  [DEBUG] Dropping completely NaN weather column: temperature for Badulla
  [DEBUG] Dropping completely NaN weather column: relative_humidity for Badulla
  [DEBUG] Dropping completely NaN weather column: rainfall for Badulla
  [DEBUG] Dropping completely NaN weather column: wind_speed for Badulla
  [DEBUG] Dropping completely NaN weather column: soil_temperature for Badulla
  [DEBUG] Dropping completely NaN weather column: soil_moisture for Badulla
  [DEBUG] Length of merged_df after dropping rows with missing price: 189
  [DEBUG] Length of merged_df after feature engineering (before final dropna): 189
  [DEBUG] Length of merged_df after dropping initial 12 rows (for lags): 177
  [DEBUG] Length of merged_df after final dropna on subset: 177
Predicted price:

/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-3053392778.py:66: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged[col] = merged[col].fillna(method='ffill').fillna(method='bfill')
/tmp/ipython-input-30533

In [22]:
import joblib

In [24]:
joblib.dump(final_global_model, "best_lgbm_model.pkl")
joblib.dump(global_imputer, "best_imputer.pkl")
joblib.dump(feature_cols_global, "feature_cols.pkl")

['feature_cols.pkl']